In [1]:
import os, random, shutil
import numpy as np
import pandas as pd
from glob import glob
from collections import defaultdict
from scipy.ndimage import label, find_objects
from scipy.spatial import cKDTree
from scipy.spatial.distance import directed_hausdorff
from skimage import measure
from sklearn.model_selection import train_test_split
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as tvm
import segmentation_models_pytorch as smp
from tqdm import tqdm
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display
from torch.utils.data import DataLoader, TensorDataset
 
IMAGES_ROOT = "../data/20_trees/Images"
GT_ROOT     = "../data/20_trees/Ground_truths"
CACHE_DIR   = "./eval_cache_3D"
OUT_DIR     = "./outputs/evaluation_3d"
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(OUT_DIR,   exist_ok=True)
 
CLASS_REMAP    = {1: 0, 2: 1, 3: 2, 4: 3, 5: 4}
CLASS_NAMES    = ["Background", "Bark", "Wood", "Knot", "Crack"]
NUM_CLASSES    = 5
DEFECT_CLASSES = [3, 4]
CLASS_COLORS_PLOTLY = {
    0: "rgb(74,85,104)",   1: "rgb(141,110,99)",
    2: "rgb(212,169,106)", 3: "rgb(255,111,0)",
    4: "rgb(229,57,53)",
}
 
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cpu


In [2]:
PREFIX_OVERRIDE = {}
 
def mask_stem_to_image_stem(mask_filename):
    name  = os.path.splitext(mask_filename)[0]
    parts = name.split("_", 2)
    stem  = parts[2] if len(parts) == 3 else name
    for wrong, correct in PREFIX_OVERRIDE.items():
        if stem.startswith(wrong):
            stem = correct + stem[len(wrong):]
            break
    return stem
 
image_paths  = sorted(glob(os.path.join(IMAGES_ROOT, "**", "*.tif"), recursive=True))
mask_paths   = sorted(glob(os.path.join(GT_ROOT, "**", "GroundTruthProject", "PixelLabelData", "*.png"), recursive=True))
image_lookup = {os.path.splitext(os.path.basename(p))[0]: p for p in image_paths}
 
all_pairs = []
for mp in mask_paths:
    stem = mask_stem_to_image_stem(os.path.basename(mp))
    if stem in image_lookup:
        all_pairs.append((image_lookup[stem], mp))
 
def get_log_id(img_path):
    parts     = img_path.replace("\\", "/").split("/")
    log_names = set(os.listdir(IMAGES_ROOT))
    return next((p for p in parts if p in log_names), None)
 
log_pairs = defaultdict(list)
for pair in all_pairs:
    log_id = get_log_id(pair[0])
    if log_id:
        log_pairs[log_id].append(pair)
 
all_logs         = sorted(log_pairs.keys())
COMMON_TEST_LOGS = ["Dub 9", "kmen9"]
test_pairs       = [p for lg in COMMON_TEST_LOGS for p in log_pairs[lg]]
print(f"Total pairs: {len(all_pairs)} | Test pairs: {len(test_pairs)}")

Total pairs: 0 | Test pairs: 0


In [3]:
PATCH_SIZE = 256
 
def crop_to_foreground(img, mask, margin=10):
    coords = np.column_stack(np.where(mask > 0))
    if coords.size == 0:
        return img, mask
    y_min, x_min = coords.min(axis=0)
    y_max, x_max = coords.max(axis=0)
    return (img[max(y_min-margin,0):min(y_max+margin,img.shape[0]),
                max(x_min-margin,0):min(x_max+margin,img.shape[1])],
            mask[max(y_min-margin,0):min(y_max+margin,mask.shape[0]),
                 max(x_min-margin,0):min(x_max+margin,mask.shape[1])])
 
def remap_mask(mask):
    out = np.zeros_like(mask, dtype=np.int64)
    for raw, idx in CLASS_REMAP.items():
        out[mask == raw] = idx
    return out
 
def load_and_preprocess(img_path, mask_path):
    img  = cv2.imread(img_path,  cv2.IMREAD_GRAYSCALE)
    mask = cv2.imread(mask_path, cv2.IMREAD_UNCHANGED)
    if mask.shape[:2] != img.shape[:2]:
        mask = cv2.resize(mask, (img.shape[1], img.shape[0]), interpolation=cv2.INTER_NEAREST)
    img, mask = crop_to_foreground(img, mask)
    img  = cv2.resize(img,  (PATCH_SIZE, PATCH_SIZE), interpolation=cv2.INTER_AREA)
    mask = cv2.resize(mask, (PATCH_SIZE, PATCH_SIZE), interpolation=cv2.INTER_NEAREST)
    img  = img.astype(np.float32)
    img  = (img - img.mean()) / (img.std() + 1e-6)
    img  = np.expand_dims(img, axis=0)
    return np.array(img, dtype=np.float32), np.array(remap_mask(mask), dtype=np.int64)
 
def build_cache(pairs, cache_dir, force=False):
    if force and os.path.exists(cache_dir):
        shutil.rmtree(cache_dir)
    os.makedirs(cache_dir, exist_ok=True)
    cached = []
    for img_path, mask_path in tqdm(pairs, desc=f"Caching"):
        stem      = os.path.splitext(os.path.basename(img_path))[0]
        img_file  = os.path.join(cache_dir, stem + "_img.npy")
        mask_file = os.path.join(cache_dir, stem + "_mask.npy")
        if not os.path.exists(img_file):
            img, mask = load_and_preprocess(img_path, mask_path)
            np.save(img_file, img); np.save(mask_file, mask)
        cached.append((img_file, mask_file))
    print(f"  OK - {len(cached)} pairs cached")
    return cached
 
cached_test = build_cache(test_pairs, os.path.join(CACHE_DIR, "test"), force=False)

Caching: 0it [00:00, ?it/s]

  OK - 0 pairs cached


In [4]:
class DilatedSegCNN(nn.Module):
    def __init__(self, in_channels=1, num_classes=NUM_CLASSES):
        super().__init__()
        self.pool = nn.MaxPool2d(2)
        self.enc1 = self._block(in_channels, 64,  dilation=1)
        self.enc2 = self._block(64,  128, dilation=2)
        self.enc3 = self._block(128, 256, dilation=4)
        self.enc4 = self._block(256, 512, dilation=8)
        self.dec3 = self._upblock(512, 256)
        self.dec2 = self._upblock(256, 128)
        self.dec1 = self._upblock(128, 64)
        self.classifier = nn.Conv2d(64, num_classes, kernel_size=1)
    def _block(self, in_c, out_c, dilation):
        return nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=dilation, dilation=dilation),
            nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=dilation, dilation=dilation),
            nn.BatchNorm2d(out_c), nn.ReLU(inplace=True))
    def _upblock(self, in_c, out_c):
        return nn.Sequential(
            nn.ConvTranspose2d(in_c, out_c, kernel_size=2, stride=2),
            nn.BatchNorm2d(out_c), nn.ReLU(inplace=True))
    def forward(self, x):
        x1=self.enc1(x); x2=self.enc2(self.pool(x1))
        x3=self.enc3(self.pool(x2)); x4=self.enc4(self.pool(x3))
        x=self.dec3(x4)+x3; x=self.dec2(x)+x2; x=self.dec1(x)+x1
        return self.classifier(x)
 
def build_unetpp():
    model = smp.UnetPlusPlus(encoder_name="resnet34", encoder_weights=None,
                              in_channels=1, classes=NUM_CLASSES, activation=None)
    enc_w = tvm.resnet34(weights=tvm.ResNet34_Weights.IMAGENET1K_V1).state_dict()
    enc_w["conv1.weight"] = enc_w["conv1.weight"].mean(dim=1, keepdim=True)
    model_dict = model.encoder.state_dict()
    matched    = {k: v for k, v in enc_w.items()
                  if k in model_dict and v.shape == model_dict[k].shape}
    model_dict.update(matched); model.encoder.load_state_dict(model_dict, strict=False)
    return model
 
def load_checkpoint(model_cfg):
    model = DilatedSegCNN() if model_cfg["type"] == "dilated" else build_unetpp()
    model.load_state_dict(torch.load(model_cfg["path"], map_location=device))
    return model.to(device).eval()

In [5]:
MODELS = {
    "DilatedCNN+MRF log-split": {
        "path": "../logs/20_trees/checkpoints/best.pt",
        "type": "dilated",
        "mrf":  True,
        "mrf_beta": 0.8,
    },
    "DilatedCNN+MRF random-split": {
        "path": "../logs/20_trees_random-split/checkpoints/best.pt",
        "type": "dilated",
        "mrf":  True,
        "mrf_beta": 0.8,
    },
    "UNet++ log-split": {
        "path": "../logs/20_trees_UNET++/checkpoints/best.pt",
        "type": "unetpp",
        "mrf":  False,
        "mrf_beta": 0.0,
    },
    "UNet++ random-split": {
        "path": "../logs/20_trees_UNET++_random-split/checkpoints/best.pt",
        "type": "unetpp",
        "mrf":  False,
        "mrf_beta": 0.0,
    },
}

COMMON_TEST_LOGS = ["Dub 9", "kmen9"]

In [6]:
def mrf_gibbs_sampling_2d(prob_map, iterations=5, beta=0.8):
    C, H, W = prob_map.shape; dev = prob_map.device
    labels  = torch.argmax(prob_map, dim=0)
    kernel  = torch.ones((C, 1, 3, 3), device=dev); kernel[:, :, 1, 1] = 0
    for _ in range(iterations):
        onehot         = F.one_hot(labels, num_classes=C).permute(2,0,1)[None].float()
        neighbor_count = F.conv2d(onehot, kernel, padding=1, groups=C).squeeze(0)
        energy         = -torch.log(prob_map + 1e-6) + beta * (8 - neighbor_count)
        labels         = torch.argmin(energy, dim=0)
    return labels
 
def mrf_gibbs_sampling_3d(prob_map, iterations=3, beta=0.8):
    """From mrf3d.py - adapted for NUM_CLASSES=5."""
    C, D, H, W = prob_map.shape; dev = prob_map.device
    labels = torch.argmax(prob_map, dim=0)
    kernel = torch.ones((1, 1, 3, 3, 3), device=dev); kernel[0,0,1,1,1] = 0
    for _ in range(iterations):
        smooth_costs = torch.zeros((C, D, H, W), device=dev)
        for cls in range(C):
            cls_mask    = (labels == cls).float().unsqueeze(0).unsqueeze(0)
            neigh_count = F.conv3d(cls_mask, kernel, padding=1).squeeze()
            smooth_costs[cls] = 26 - neigh_count
        energy = -torch.log(prob_map + 1e-6) + beta * smooth_costs
        labels = torch.argmin(energy, dim=0)
    return labels

In [7]:
APPLY_3D_MRF = True   # set False to use 2D predictions only
 
def build_log_volumes(model_cfg, apply_3d_mrf=APPLY_3D_MRF):
    model   = load_checkpoint(model_cfg)
    use_mrf = model_cfg["mrf"]; beta = model_cfg.get("mrf_beta", 0.8)
 
    log_to_indices = defaultdict(list)
    for i, pair in enumerate(test_pairs):
        log_to_indices[get_log_id(pair[0])].append(i)
 
    gt_vols = {}; pred_vols = {}; prob_vols = {}
 
    for log_id, indices in log_to_indices.items():
        imgs  = np.stack([np.load(cached_test[i][0]) for i in indices])
        masks = np.stack([np.load(cached_test[i][1]) for i in indices])
        ds     = TensorDataset(torch.tensor(imgs, dtype=torch.float32),
                               torch.tensor(masks, dtype=torch.long))
        loader = DataLoader(ds, batch_size=16, shuffle=False, num_workers=0)
        slice_probs = []; slice_preds = []
 
        with torch.no_grad():
            for imgs_b, _ in tqdm(loader, desc=f"  {log_id}", leave=False):
                probs_b = F.softmax(model(imgs_b.to(device)), dim=1)
                if use_mrf:
                    preds_b = torch.stack([mrf_gibbs_sampling_2d(probs_b[b], beta=beta)
                                           for b in range(len(probs_b))]).cpu()
                else:
                    preds_b = torch.argmax(probs_b, dim=1).cpu()
                slice_probs.append(probs_b.cpu()); slice_preds.append(preds_b)
 
        gt_vol   = masks
        pred_vol = torch.cat(slice_preds, dim=0).numpy()
        prob_vol = torch.cat(slice_probs, dim=0).numpy().transpose(1, 0, 2, 3)  # (C,D,H,W)
 
        if apply_3d_mrf:
            print(f"    Applying 3D MRF on {log_id}...")
            refined  = mrf_gibbs_sampling_3d(
                torch.tensor(prob_vol, dtype=torch.float32, device=device),
                iterations=3, beta=0.5
            )
            pred_vol = refined.cpu().numpy()
 
        gt_vols[log_id] = gt_vol; pred_vols[log_id] = pred_vol; prob_vols[log_id] = prob_vol
 
    del model; torch.cuda.empty_cache()
    return gt_vols, pred_vols, prob_vols

In [8]:
def safe_compactness(region_mask):
    try:
        verts, _, _, _ = measure.marching_cubes(region_mask.astype(np.float32), level=0.5)
        return (float(verts.shape[0]) ** 1.5) / (float(np.count_nonzero(region_mask)) + 1e-6)
    except:
        return None
 
def compute_3d_boundary_metrics(pred_bin, gt_bin):
    pred_pts = np.column_stack(np.where(pred_bin))
    gt_pts   = np.column_stack(np.where(gt_bin))
    if len(pred_pts) == 0 or len(gt_pts) == 0:
        return float("nan"), float("nan")
    hd   = max(directed_hausdorff(pred_pts, gt_pts)[0],
               directed_hausdorff(gt_pts, pred_pts)[0])
    assd = (cKDTree(gt_pts).query(pred_pts)[0].mean() +
            cKDTree(pred_pts).query(gt_pts)[0].mean()) / 2
    return hd, assd
 
def compute_volume_metrics_3d(pred_vol, gt_vol, class_idx):
    pred_bin = (pred_vol == class_idx); gt_bin = (gt_vol == class_idx)
    tp = int((pred_bin & gt_bin).sum())
    fp = int((pred_bin & ~gt_bin).sum())
    fn = int((~pred_bin & gt_bin).sum())
    iou       = tp/(tp+fp+fn)     if (tp+fp+fn)>0 else float("nan")
    dice      = 2*tp/(2*tp+fp+fn) if (2*tp+fp+fn)>0 else float("nan")
    recall    = tp/(tp+fn)        if (tp+fn)>0 else float("nan")
    precision = tp/(tp+fp)        if (tp+fp)>0 else float("nan")
    vs        = 1-abs(pred_bin.sum()-gt_bin.sum())/(pred_bin.sum()+gt_bin.sum()+1e-9)
    hd, assd  = (compute_3d_boundary_metrics(pred_bin, gt_bin)
                 if (gt_bin.sum()>0 and pred_bin.sum()>0) else (float("nan"), float("nan")))
    n_vox     = int(pred_bin.sum())
    labeled, n_comp = label(pred_bin)
    slices    = find_objects(labeled)
    comp_sizes  = [int((labeled[s]==(i+1)).sum()) for i,s in enumerate(slices)]
    compacts    = [c for c in [safe_compactness((labeled[s]==(i+1))) for i,s in enumerate(slices)] if c]
    continuity  = max(comp_sizes)/(n_vox+1e-9) if comp_sizes else float("nan")
    compactness = float(np.mean(compacts)) if compacts else float("nan")
    return {"iou": iou, "dice": dice, "recall": recall, "precision": precision,
            "vs": vs, "hd_3d": hd, "assd_3d": assd, "volume_vox": n_vox,
            "components": n_comp, "continuity": continuity, "compactness": compactness}

In [9]:
all_3d_results = {}
 
for model_name, model_cfg in MODELS.items():
    if not os.path.exists(model_cfg["path"]):
        print(f"Skipping '{model_name}' - not found: {model_cfg['path']}"); continue
    print(f"\n{'─'*60}\n Evaluating 3D: {model_name}\n{'─'*60}")
    gt_vols, pred_vols, prob_vols = build_log_volumes(model_cfg)
    log_metrics = {}
    for log_id in gt_vols:
        log_metrics[log_id] = {
            cls_name: compute_volume_metrics_3d(pred_vols[log_id], gt_vols[log_id], c)
            for c, cls_name in enumerate(CLASS_NAMES)
        }
        print(f"  {log_id} done")
    all_3d_results[model_name] = {"metrics": log_metrics,
                                   "gt_vols": gt_vols, "pred_vols": pred_vols}
 
print("\nAll 3D evaluations complete.")


────────────────────────────────────────────────────────────
 Evaluating 3D: DilatedCNN+MRF log-split
────────────────────────────────────────────────────────────

────────────────────────────────────────────────────────────
 Evaluating 3D: DilatedCNN+MRF random-split
────────────────────────────────────────────────────────────

────────────────────────────────────────────────────────────
 Evaluating 3D: UNet++ log-split
────────────────────────────────────────────────────────────
Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to C:\Users\tomca/.cache\torch\hub\checkpoints\resnet34-b627a593.pth


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 83.3M/83.3M [00:03<00:00, 26.0MB/s]



────────────────────────────────────────────────────────────
 Evaluating 3D: UNet++ random-split
────────────────────────────────────────────────────────────

All 3D evaluations complete.


In [10]:
REPORT_METRICS = ["iou","dice","recall","precision","vs","hd_3d","assd_3d"]
METRIC_FMT     = {"iou":".4f","dice":".4f","recall":".4f","precision":".4f",
                  "vs":".4f","hd_3d":".2f","assd_3d":".2f"}
 
print(f"\n{'='*100}")
print(f"  3D MODEL COMPARISON - Test logs: {COMMON_TEST_LOGS}")
print(f"{'='*100}")
 
for cls_name in ["Knot","Crack"]:
    print(f"\n-- {cls_name} {'─'*80}")
    col_w = max(len(n) for n in all_3d_results)+2
    print(f"  {'Model':<{col_w}} {'Log':<12}" + "".join(f"{m:>11}" for m in REPORT_METRICS))
    print("  "+"─"*(col_w+12+11*len(REPORT_METRICS)))
    for model_name, res in all_3d_results.items():
        for log_id in COMMON_TEST_LOGS:
            if log_id not in res["metrics"]: continue
            m    = res["metrics"][log_id][cls_name]
            line = f"  {model_name:<{col_w}} {log_id:<12}"
            for metric in REPORT_METRICS:
                v = m[metric]
                line += f"{'n/a':>11}" if np.isnan(v) else f"{v:{METRIC_FMT[metric]}:>11}"
            print(line)
 
    print(f"\n  Volume metrics - {cls_name}:")
    print(f"  {'Model':<{col_w}} {'Log':<12} {'Vol(vox)':>10} {'Comps':>8} {'Continuity':>12} {'Compactness':>13}")
    for model_name, res in all_3d_results.items():
        for log_id in COMMON_TEST_LOGS:
            if log_id not in res["metrics"]: continue
            m = res["metrics"][log_id][cls_name]
            print(f"  {model_name:<{col_w}} {log_id:<12}"
                  f" {m['volume_vox']:>10,} {m['components']:>8}"
                  f" {m['continuity']:>12.4f} {m['compactness']:>13.2f}")


  3D MODEL COMPARISON - Test logs: ['Dub 9', 'kmen9']

-- Knot ────────────────────────────────────────────────────────────────────────────────
  Model                         Log                 iou       dice     recall  precision         vs      hd_3d    assd_3d
  ──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

  Volume metrics - Knot:
  Model                         Log            Vol(vox)    Comps   Continuity   Compactness

-- Crack ────────────────────────────────────────────────────────────────────────────────
  Model                         Log                 iou       dice     recall  precision         vs      hd_3d    assd_3d
  ──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

  Volume metrics - Crack:
  Model                         Log            Vol(vox)    Comps   Continuity   Compactness


In [11]:
def make_mesh_traces(volume_3d, spacing=(10,1,1), opacity=0.15, show_legend=True):
    traces = []; volume_3d = volume_3d.astype(np.uint8)
    for c, cls_name in enumerate(CLASS_NAMES):
        if c == 0: continue
        mask = (volume_3d != 0) if c == 1 else (volume_3d == c)
        if np.count_nonzero(mask) == 0: continue
        try:
            verts, faces, _, _ = measure.marching_cubes(mask.astype(np.float32), level=0.5, spacing=spacing)
            traces.append(go.Mesh3d(
                x=verts[:,0], y=verts[:,1], z=verts[:,2],
                i=faces[:,0], j=faces[:,1], k=faces[:,2],
                color=CLASS_COLORS_PLOTLY[c], opacity=opacity, name=cls_name,
                flatshading=True, lighting=dict(ambient=0.5, diffuse=0.5),
                legendgroup=f"class_{c}", showlegend=show_legend,
            ))
        except Exception as e:
            print(f"  Skipping {cls_name}: {e}")
    return traces
 
def visualise_3d(log_id, gt_vol, pred_vols_dict, spacing=(10,1,1), defect_only=False):
    col_titles = ["Ground Truth"] + list(pred_vols_dict.keys())
    n_cols     = len(col_titles)
    fig = make_subplots(rows=1, cols=n_cols,
                        specs=[[{"type":"mesh3d"}]*n_cols],
                        subplot_titles=col_titles, horizontal_spacing=0.02)
    scene_cfg = dict(aspectmode="data", camera=dict(eye=dict(x=-3.2, y=-4, z=0.8)))
 
    def get_traces(vol):
        if defect_only:
            traces = []
            for c in [3,4]:
                mask = (vol == c)
                if np.count_nonzero(mask)==0: continue
                try:
                    verts, faces, _, _ = measure.marching_cubes(mask.astype(np.float32), level=0.5, spacing=spacing)
                    traces.append(go.Mesh3d(
                        x=verts[:,0], y=verts[:,1], z=verts[:,2],
                        i=faces[:,0], j=faces[:,1], k=faces[:,2],
                        color=CLASS_COLORS_PLOTLY[c], opacity=0.7,
                        name=CLASS_NAMES[c], flatshading=True,
                        legendgroup=f"class_{c}", showlegend=True))
                except: pass
            return traces
        return make_mesh_traces(vol, spacing=spacing)
 
    for trace in get_traces(gt_vol):
        fig.add_trace(trace, row=1, col=1)
    for col_i,(model_name,pred_vol) in enumerate(pred_vols_dict.items(), start=2):
        for trace in get_traces(pred_vol):
            trace.showlegend = False
            fig.add_trace(trace, row=1, col=col_i)
    for col_i in range(1, n_cols+1):
        fig.update_layout(**{("scene" if col_i==1 else f"scene{col_i}"): scene_cfg})
 
    tag = "Defect-only" if defect_only else "Full"
    fig.update_layout(
        title=dict(text=f"3D {tag} Comparison - {log_id}", font=dict(size=16)),
        height=600, margin=dict(l=0,r=0,b=0,t=60),
        showlegend=True, legend_itemclick="toggle", legend_itemdoubleclick="toggleothers",
        uirevision="constant",
    )
    return fig
 
 
for log_id in COMMON_TEST_LOGS:
    pred_vols_for_log = {mn: res["pred_vols"][log_id]
                         for mn, res in all_3d_results.items()
                         if log_id in res["pred_vols"]}
    gt_vol = list(all_3d_results.values())[0]["gt_vols"][log_id]
 
    # Full segmentation view
    print(f"\nRendering full 3D for {log_id}...")
    fig = visualise_3d(log_id, gt_vol, pred_vols_for_log)
    display(fig)
    out = os.path.join(OUT_DIR, f"3d_full_{log_id.replace(' ','_')}.html")
    fig.write_html(out); print(f"Saved -> {out}")
 
    # Defect-only view
    print(f"Rendering defect-only 3D for {log_id}...")
    fig2 = visualise_3d(log_id, gt_vol, pred_vols_for_log, defect_only=True)
    display(fig2)
    out2 = os.path.join(OUT_DIR, f"3d_defects_{log_id.replace(' ','_')}.html")
    fig2.write_html(out2); print(f"Saved -> {out2}")

KeyError: 'Dub 9'